[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C03/E03-2_Qualite_Surface_Usinee.ipynb)

# État de surface d'une pièce usinée : mesurer les liens entre variables

## Le contexte

En finition, la qualité d'une surface usinée se juge à sa rugosité arithmétique Ra, en micromètres. Elle dépend des réglages de coupe (vitesse, avance, profondeur de passe, lubrification), de la matière (dureté, nature) et de l'état de l'outil. Mesurer Ra demande un rugosimètre et du temps ; la prédire avant l'usinage permettrait de régler la machine du premier coup.

Le relevé contient aussi des indicateurs d'énergie de coupe et d'échauffement, dont plusieurs se ressemblent beaucoup. Avant de modéliser, il faut mesurer les liens : lesquelles des grandeurs agissent sur Ra, lesquelles ne font que se répéter entre elles.

C'est un problème de régression : la grandeur à prédire est un nombre, en micromètres.

Défi : prédire `rugosite_ra_um` et obtenir un R² supérieur à celui du modèle de départ sur la base de test. Ce modèle de départ n'utilise qu'une seule variable, l'avance par tour, et atteint un R² de 0,915 : à vous d'ajouter les autres. La règle de découpage est fixée plus bas. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Qualite_Surface_Usinee.xlsx`, dans le dossier `donnees` du dépôt : 178 pièces, 14 colonnes. Une ligne est une pièce usinée.

| Colonne | Signification | Unité |
|---|---|---|
| `classe_materiau` | matière : `Acier_Doux`, `Alliage_Aluminium` ou `Titane_Grade5` | catégorie |
| `vitesse_coupe_m_min` | vitesse de coupe | m/min |
| `avance_mm_tr` | avance par tour | mm/tr |
| `profondeur_passe_mm` | profondeur de passe | mm |
| `durete_matiere_HB` | dureté de la matière | HB |
| `taux_lubrifiant_pourcent` | taux de lubrifiant | % |
| `taux_usure_outil_pourcent` | usure de l'outil | % |
| `energie_specifique_MJ_mm3` | énergie spécifique de coupe | MJ/mm³ |
| `energie_specifique_ajustee` | énergie spécifique corrigée, construite | sans unité |
| `intensite_thermique`, `indice_stabilite_thermique` | indicateurs d'échauffement, construits | sans unité |
| `indice_energie_globale` | indicateur d'énergie globale, construit | sans unité |
| `rugosite_ra_um` | cible : rugosité arithmétique | µm |
| `classe_qualite_surface` | classe de qualité : `Faible`, `Moyenne` ou `Élevée` | catégorie |

Origine : jeu public transformé. Ce support pédagogique réinterprète les mesures d'origine en réglages de coupe ; la rugosité, sa classe et les indices ont été construits. Il enseigne une méthode, et aucune conclusion d'usinage n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions et ses cases vides avant toute modélisation. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Qualite_Surface_Usinee.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Qualite_Surface_Usinee.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "rugosite_ra_um"
print("dimensions :", df.shape)
print("cases vides au total :", int(df.isna().sum().sum()))
print(df["classe_materiau"].value_counts().to_string())
print(df[cible].describe().round(2).to_string())
df.head(3)

Lecture du résultat : 178 pièces, 14 colonnes, aucune case vide. La rugosité va de 0,66 à 4,98 µm, pour une moyenne de 2,62 µm.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Qualite_Surface_Usinee.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Étape 2 : séparer l'apprentissage et le test

Un quart des pièces est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la rugosité. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. La structure du fichier se lit sur tout le classeur ; tout ce qui croise la rugosité, dans les étapes suivantes, se lit sur l'apprentissage seul.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=cible)
y = df[cible]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE)
print("apprentissage :", len(X_train), "pièces")
print("test :", len(X_test), "pièces")

### Étape 3 : écarter ce qui se déduit de la cible

La classe de qualité semble résumer la rugosité. On vérifie, classe par classe, le minimum et le maximum de Ra sur l'apprentissage, puis on retire la classe des entrées, de l'apprentissage comme du test.

In [ ]:
appr = df.loc[X_train.index]
print(appr.groupby("classe_qualite_surface")[cible].agg(["min", "max", "count"]).round(3).to_string())
X_train = X_train.drop(columns="classe_qualite_surface")
X_test = X_test.drop(columns="classe_qualite_surface")

Lecture du résultat : les trois classes se rangent par tranches de Ra qui ne se chevauchent pas : la classe est déduite de la rugosité mesurée. Elle ne peut pas servir à la prédire, puisqu'elle n'existe qu'une fois Ra connue : elle est écartée des entrées.

### Étape 4 : mesurer les liens avec la cible

Deux coefficients mesurent un lien. Celui de Pearson mesure la part linéaire : il vaut 1 ou -1 si les points sont alignés. Celui de Spearman compare les rangs : il détecte tout lien qui va toujours dans le même sens, droit ou courbe. Un écart notable entre les deux signale une courbure ou des points atypiques. On calcule les deux pour chaque grandeur numérique de l'apprentissage, puis on trace Ra en fonction de l'avance.

In [ ]:
entrees = X_train.drop(columns="classe_materiau")
liens = pd.DataFrame({
    "Pearson": entrees.corrwith(y_train),
    "Spearman": entrees.corrwith(y_train, method="spearman")})
print(liens.sort_values("Pearson", ascending=False).round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for matiere, marqueur, gris in [("Acier_Doux", "o", "0.65"), ("Alliage_Aluminium", "s", "0.35"), ("Titane_Grade5", "^", "0.0")]:
    pieces = appr[appr["classe_materiau"] == matiere]
    ax.scatter(pieces["avance_mm_tr"], pieces[cible], s=22, marker=marqueur, color=gris, label=matiere)
ax.set_xlabel("avance par tour (mm/tr)")
ax.set_ylabel("rugosité Ra (µm)")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()

Lecture du résultat : l'avance domine, avec 0,94 en Pearson et 0,94 en Spearman. La profondeur de passe vient loin derrière (0,43) ; les autres grandeurs sont faiblement liées à Ra, en apparence. Le nuage est presque une droite, que l'on distingue bien sur le graphique ; les trois matières suivent la même tendance, avec des avances plus grandes pour le titane. Les deux coefficients sont proches : le lien est bien linéaire, sans valeur extrême qui l'abîme.

### Étape 5 : mesurer les liens entre les entrées

Deux entrées très corrélées entre elles disent presque la même chose. Pour les repérer, on représente la valeur absolue de la corrélation de chaque paire d'entrées de l'apprentissage (plus la case est sombre, plus le lien est fort), puis on liste les paires au-delà de 0,85. Choisir les colonnes à retirer est une décision apprise sur les données : elle se prend sur l'apprentissage seul.

In [ ]:
correlations = entrees.corr().abs()
fig, ax = plt.subplots(figsize=(6.5, 5.5))
image = ax.imshow(correlations, cmap="gray_r", vmin=0, vmax=1)
ax.set_xticks(range(len(correlations)))
ax.set_yticks(range(len(correlations)))
ax.set_xticklabels(range(1, len(correlations) + 1), fontsize=8)
ax.set_yticklabels([f"{k + 1}. {nom}" for k, nom in enumerate(correlations.columns)], fontsize=7)
fig.colorbar(image, ax=ax, label="valeur absolue de la corrélation")
plt.show()

paires = [(a, b, correlations.loc[a, b]) for k, a in enumerate(correlations.columns)
          for b in correlations.columns[k + 1:] if correlations.loc[a, b] > 0.85]
for a, b, r in sorted(paires, key=lambda t: -t[2]):
    print(f"{a} et {b} : {r:.2f}")

Lecture du résultat : 3 paires dépassent 0,85. Les deux énergies spécifiques se répètent presque exactement, comme les deux indicateurs d'échauffement, qui varient en sens contraire ; le taux de lubrifiant et l'usure de l'outil vont ensemble. Ces colonnes sont redondantes : elles ne portent pas d'information indépendante. Un modèle linéaire qui les reçoit toutes peut répartir un même effet entre elles de façon instable, et ses coefficients deviennent illisibles.

### Étape 6 : construire le modèle de départ

Le modèle de départ est volontairement simple : une régression linéaire qui cherche la droite minimisant la somme des carrés des écarts, avec la seule variable la plus liée à Ra, l'avance par tour. Il apprend sur `X_train` seul. Les autres colonnes sont laissées de côté : elles sont votre marge de progression.

In [ ]:
variable = "avance_mm_tr"

# Votre code :

### Étape 7 : mesurer sur la base de test

Le modèle prédit les pièces qu'il n'a jamais vues. Trois indicateurs sont lus : le coefficient de détermination R², et deux erreurs en micromètres, la racine de l'erreur quadratique moyenne (RMSE) et l'erreur absolue moyenne (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

y_pred = modele.predict(X_test[[variable]])
score_depart = r2_score(y_test, y_pred)
print(f"R² : {score_depart:.3f}")
print(f"RMSE : {root_mean_squared_error(y_test, y_pred):.3f} µm")
print(f"MAE : {mean_absolute_error(y_test, y_pred):.3f} µm")
print(f"pente : {modele.coef_[0]:.2f} µm par mm/tr d'avance")

Lecture du résultat : avec la seule avance, le R² est d'environ 0,91, et l'erreur absolue moyenne de 0,20 µm pour une rugosité moyenne de 2,52 µm. La pente, d'environ 12 µm par mm/tr, se lit en unités physiques : augmenter l'avance de 0,1 mm/tr ajoute environ 1,2 µm de rugosité. C'est ce chiffre, mesuré sur des pièces que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : garder les entrées indépendantes

Le modèle de départ de l'étape 6 reste le score de référence du défi. Les deux cellules suivantes mesurent d'abord la redondance des entrées par le facteur d'inflation de la variance (VIF), puis comparent par validation croisée le modèle à toutes les entrées et le modèle qui n'en garde qu'une par groupe de colonnes redondantes. Le test ne sert qu'une fois. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

## Le score de référence

Le score publié est le R² du modèle de départ de l'étape 6 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = r2_score(y_test, modele.predict(X_test[[variable]]))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE r2 = %.3f" % score)

## Synthèse

1. Une colonne déduite de la cible, comme la classe de qualité, est écartée avant tout calcul de lien.
2. Pearson mesure le lien linéaire, Spearman le lien monotone ; deux valeurs proches indiquent un lien droit, un grand écart une courbure ou des points atypiques.
3. Les liens entre entrées comptent autant que les liens avec la cible : deux colonnes très corrélées se répètent, et le VIF chiffre cette répétition.
4. Retirer, dans chaque groupe redondant, la colonne la moins liée à la cible simplifie le modèle sans le dégrader.
5. Une pente se lit en unités physiques, avec prudence : elle décrit ces données, dans ce domaine.